# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [3]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

# Step 2: Build your chart
# YOUR CODE HERE
# Step 2: Build your chart
highlight = 'South Asia'

# Add the global average as its own row, then re-sort so it lands in the right place
global_row = pd.DataFrame({'Region': ['Global average'],
                           'Happiness_Score': [df['Happiness_Score'].mean()]})
plot_df = (
    pd.concat([region_avg[['Region', 'Happiness_Score']], global_row], ignore_index=True)
      .sort_values('Happiness_Score')
)

def pick_colour(region):
    if region == highlight:
        return 'highlight'
    if region == 'Global average':
        return 'average'
    return 'other'

plot_df['colour'] = plot_df['Region'].apply(pick_colour)

fig = px.bar(
    plot_df,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    color='colour',
    color_discrete_map={'highlight': '#D1495B', 'average': '#2E4057', 'other': '#B8BEC6'},
    text='Happiness_Score',
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside', cliponaxis=False)

fig.update_layout(
    title=dict(
        text='South Asia is the least happy region, scoring about half of Western Europe',
        x=0.08, xanchor='left',
    ),
    xaxis=dict(title='Average happiness score (0–10)', range=[0, 8], showgrid=False),
    yaxis=dict(title='', ticksuffix='  ', categoryorder='array',
               categoryarray=plot_df['Region']),
    showlegend=False,
    plot_bgcolor='white',
    margin=dict(l=10, r=40, t=90, b=40),
    height=480,
)

fig.show()


                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
# Step 2: Build your chart
top_colour, bottom_colour = '#2A9D8F', '#D1495B'

fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={'Top 8': top_colour, 'Bottom 8': bottom_colour},
    text='Happiness_Score',
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside', cliponaxis=False)

# Visual separator: dashed line between the two groups (bars 0-7 are the bottom 8)
fig.add_hline(y=7.5, line_dash='dash', line_color='#888', line_width=1)

# Annotation that states the size of the gap
top_mean = top8['Happiness_Score'].mean()
bottom_mean = bottom8['Happiness_Score'].mean()
fig.add_annotation(
    x=8.6, y=7.5, xanchor='left', showarrow=False, align='left',
    text=f'<b>Gap: {top_mean - bottom_mean:.1f} points</b><br>'
         f'Top 8 average {top_mean:.2f}<br>Bottom 8 average {bottom_mean:.2f}',
    font=dict(size=12, color='#333'), bgcolor='white', borderpad=4,
)

# Stretch goal: global average reference line
fig.add_vline(
    x=global_avg, line_dash='dot', line_color='#555', line_width=1.5,
    annotation_text=f'Global average {global_avg:.2f}',
    annotation_position='top',
    annotation_font=dict(size=11, color='#555'),
)

# Group labels in place of a legend
fig.add_annotation(x=8.6, y=15, xanchor='left', showarrow=False,
                   text='<b>Top 8</b>', font=dict(color=top_colour, size=13))
fig.add_annotation(x=8.6, y=0, xanchor='left', showarrow=False,
                   text='<b>Bottom 8</b>', font=dict(color=bottom_colour, size=13))

fig.update_layout(
    title=dict(
        text='The happiest countries score more than twice as high as the least happy',
        x=0, xanchor='left',
    ),
    xaxis=dict(title='Happiness score (0–10)', range=[0, 10.5], showgrid=False,
               tickvals=[0, 2, 4, 6, 8]),
    yaxis=dict(title='', ticksuffix='  ', categoryorder='array',
               categoryarray=combined['Country']),
    showlegend=False,
    plot_bgcolor='white',
    margin=dict(l=10, r=20, t=100, b=40),
    height=620,
)

fig.show()



Global average: 5.81


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [5]:
# Stretch goal — grouped bar chart
# Task 3: GDP vs. Freedom across five regions
regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']
factors = ['GDP', 'Freedom']

# GDP (log scale, about 7-11) and Freedom (0-1) use different units, so rescale
# each to 0-100 across all countries: 0 = lowest country, 100 = highest country
scaled = df[['Region'] + factors].copy()
for f in factors:
    scaled[f] = (df[f] - df[f].min()) / (df[f].max() - df[f].min()) * 100

task3 = (
    scaled[scaled['Region'].isin(regions)]
      .groupby('Region')[factors].mean()
      .sort_values('GDP', ascending=False)
      .reset_index()
      .melt(id_vars='Region', var_name='Factor', value_name='Score')
)
task3['Factor'] = task3['Factor'].replace({'GDP': 'GDP per capita'})

fig = px.bar(
    task3,
    x='Score',
    y='Region',
    orientation='h',
    color='Factor',
    barmode='group',
    color_discrete_map={'GDP per capita': '#B8BEC6', 'Freedom': '#2A9D8F'},
    text='Score',
)

fig.update_traces(texttemplate='%{text:.0f}', textposition='outside', cliponaxis=False)

fig.update_layout(
    title=dict(
        text='East Asia is the only region where freedom lags behind wealth'
             '<br><sup>Regional average of GDP per capita and freedom to make life choices, '
             'each rescaled to 0–100 (lowest to highest country)</sup>',
        x=0, xanchor='left',
    ),
    xaxis=dict(title='Score (0–100)', range=[0, 100], showgrid=False),
    # richest region at the top; reversed legend order matches the bar order
    yaxis=dict(title='', ticksuffix='  ', categoryorder='array',
               categoryarray=task3['Region'].unique()[::-1]),
    legend=dict(title='', orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0,
                traceorder='reversed'),
    plot_bgcolor='white',
    margin=dict(l=10, r=40, t=110, b=40),
    height=520,
)

fig.show()

